In [3]:
from admcycles import *

In [2]:
def make_glu_graph(g, i, n=1):
    """
    Constructs the StableGraph Gi with i elliptic tails efficiently.
    """
    if i == 0:
        return StableGraph([g], [list(range(1, n + 1))], [])
    
    # V: Main vertex (genus g-i) + i tails (genus 1)
    V = [g - i] + [1] * i
    # H: Main legs (1..n + internal) + Tail legs
    H = [list(range(1, n + i + 1))] + [[n + i + k] for k in range(1, i + 1)]
    # E: Edges connecting Main(n+k) -- Tail(n+i+k)
    E = [(n + k, n + i + k) for k in range(1, i + 1)]
    
    return StableGraph(V, H, E)

In [4]:
### WORKSPACE 1: What does FZsimplify do? ###

g = 7
n = 1

# Create an array of Graphs G where G[i] has i elliptic tails
# ranges from 0 to g
G = [make_glu_graph(g, i, n) for i in range(g + 1)]

# Create an array of Tautological Classes GLU where GLU[i] corresponds to G[i]
GLU = [graph.to_tautological_class() for graph in G]

(GLU[3]*GLU[4]).simplify()

Graph :      [1, 1, 1, 1, 3] [[6], [8], [10], [12], [1, 7, 9, 11, 13]] [(6, 7), (8, 9), (10, 11), (12, 13)]
Polynomial : -24*psi_7*psi_9*psi_11 - 24*psi_6*psi_8*psi_10 - 72*psi_6*psi_8*psi_11 - 72*psi_6*psi_9*psi_11

Graph :      [1, 1, 1, 1, 1, 2] [[7], [9], [11], [13], [15], [1, 8, 10, 12, 14, 16]] [(7, 8), (9, 10), (11, 12), (13, 14), (15, 16)]
Polynomial : 36*psi_8*psi_10 + 36*psi_7*psi_9 + 72*psi_7*psi_10

Graph :      [1, 1, 1, 1, 1, 1, 1] [[8], [10], [12], [14], [16], [18], [1, 9, 11, 13, 15, 17, 19]] [(8, 9), (10, 11), (12, 13), (14, 15), (16, 17), (18, 19)]
Polynomial : -12*psi_9 - 12*psi_8

Graph :      [1, 1, 1, 1, 1, 1, 0, 1] [[9], [11], [13], [15], [17], [19], [1, 10, 12, 14, 16, 18, 20, 21], [22]] [(9, 10), (11, 12), (13, 14), (15, 16), (17, 18), (19, 20), (21, 22)]
Polynomial : 1

In [ ]:
from admcycles import *
from itertools import product
from math import factorial

def glu_bicolored_sum(g, total_tails, bicolored_indices, n=1, extra_decorations=None):
    """
    Constructs a sum of boundary pushforwards for a graph with total_tails elliptic tails,
    where the specified bicolored_indices edges are decorated with (-psi_star - psi_dot).
    
    Args:
        g: Total genus
        total_tails: Number of elliptic tails (determines the graph Gi)
        bicolored_indices: List of indices (0 to total_tails-1) of edges to bicolor.
        n: Number of legs on the main vertex.
        extra_decorations: Optional list of TautologicalClasses to multiply with the 
                           bicolored decorations (v0 is at index 0, tails at 1..total_tails).
    """
    # 1. Create the base Graph mapping Gi
    # V: Main vertex (genus g - total_tails) + total_tails (genus 1)
    V = [g - total_tails] + [1] * total_tails
    # H: Main legs (1..n + internal) + Tail legs
    H = [list(range(1, n + total_tails + 1))] + [[n + total_tails + k] for k in range(1, total_tails + 1)]
    # E: Edges connecting Main(n+k) -- Tail(n+total_tails+k)
    E = [(n + k, n + total_tails + k) for k in range(1, total_tails + 1)]
    
    G = StableGraph(V, H, E)
    
    # 2. Iterate over bit-vectors for bicolored edges
    # For each bicolored edge, we pick either 0 (psi_star) or 1 (psi_dot)
    # The term is (-psi_star - psi_dot), so for m edges we have a global sign (-1)^m
    # and we sum over choices of psi_star or psi_dot for each edge.
    
    m = len(bicolored_indices)
    result = 0 * G.to_tautological_class()
    
    # Global sign for the product of m terms (-psi - psi)
    global_sign = (-1)**m
    
    for bits in product((0, 1), repeat=m):
        # bits list corresponds to bicolored_indices
        
        # Initialize A with extra_decorations or fundclasses
        if extra_decorations is None:
            A = [fundclass(V[v], len(H[v])) for v in range(len(V))]
        else:
            # Copy extra_decorations to avoid mutating the input
            A = list(extra_decorations)
            
        # Apply decorations from bicolored edges
        for i, bit in enumerate(bits):
            edge_idx = bicolored_indices[i]
            # star side is on vertex 0, leg is n + 1 + edge_idx
            # dot side is on vertex edge_idx + 1, leg is n + total_tails + 1 + edge_idx
            
            if bit == 0: # psi_star
                leg_star = n + 1 + edge_idx
                # Multiply the class in A[0]
                A[0] = A[0] * psiclass(leg_star, V[0], len(H[0]))
            else: # psi_dot
                # Tail vertex index is edge_idx + 1
                v_tail = edge_idx + 1
                # Tail usually has only 1 leg (the one from the edge)
                A[v_tail] = A[v_tail] * psiclass(1, 1, 1)
        
        # Accumulate the pushforward
        result += global_sign * G.boundary_pushforward(A)
        
    return result

In [ ]:
### WORKSPACE 2: Testing glu_helper ###

g = 5
n = 1

# Create an array of Graphs G where G[i] has i elliptic tails
# ranges from 0 to g
G = [make_glu_graph(g, i, n) for i in range(g + 1)]

# Create an array of Tautological Classes GLU where GLU[i] corresponds to G[i]
GLU = [graph.to_tautological_class() for graph in G]

# Check the first relation
rel1 = (GLU[2]*GLU[3] - (6*glu_bicolored_sum(g, 3, [0, 1]) + 6*glu_bicolored_sum(g, 4, [0]) + GLU[5]))
print(f"Is the first relation zero? {rel1.is_zero()}")

g = 6

G = [make_glu_graph(g, i, n) for i in range(g + 1)]

GLU = [graph.to_tautological_class() for graph in G]

# Print the second output
print("\nOutput of GLU[3]^2:")
print((GLU[3]**2).FZsimplify())
print("\n===================")

# print("\nOutput of glu_bicolored_sum(g,3,[0,1,2]):")
# print(glu_bicolored_sum(g,3,[0,1,2]))


# print("\nOutput of glu_bicolored_sum(g,4,[0,1]):")
# print(glu_bicolored_sum(g,4,[0,1]))


# print("\nOutput of glu_bicolored_sum(g,5,[0]):")
# print(glu_bicolored_sum(g,5,[0]))
# print("\n===================")


# H = StableGraph([3, 1, 1, 1], [[1, 2, 3, 4], [5], [6], [7]], [(2, 5), (3, 6), (4, 7)])
# # 2. Define the factors: each edge gets a (-psi_star - psi_dot)
# psi_factors = [
#     [(2, -1), (5, -1)], # Edge (2,5): -psi_2 - psi_5
#     [(3, -1), (6, -1)], # Edge (3,6): -psi_3 - psi_6
#     [(4, -1), (7, -1)]  # Edge (4,7): -psi_4 - psi_7
# ]
# # 3. Construct the class with the factor of 6
# decorH = 6 * decorate_graph(H, psi_factors)
# print("\nDecoration of T1:")
# print(decorH)

# temp_prod=GLU[3]^2-decorH
# print("\nG3square-Hdecor-GLU[g]:")
# print(temp_prod)

# print("\nIs decorH the same as glu_bicolored with the coefficient?")
# print(f"Are they equal? {(decorH-6*glu_bicolored_sum(g,3,[0,1,2]))==0}")
# print("\n===================")

# print("\nTrying out glu_overlap_sum and removing the second term from sum:")
# temp2_prod=temp_prod-glu_overlap_sum(g=6, i=3, j=3, k=2)
# print(temp2_prod)
# print("\n===================")
# # full_relation = (GLU[3]^2-(6*glu_bicolored_sum(g,3,[0,1,2])+18*glu_bicolored_sum(g,4,[0,1])+6*glu_bicolored_sum(g,5,[0])+GLU[g]))
# # print(f"Is the full relation zero? {full_relation.is_zero()}")


Is the first relation zero? True

Output of GLU[3]^2:


In [63]:
from itertools import product

def pslambdaclass_term(d, g, n, i):
    """
    Returns the i-th term of the pslambdaclass expansion efficiently.
    """
    if i == 0:
        return lambdaclass(d, g, n)
        
    G = make_glu_graph(g, i, n)
    
    # Main decoration: Lambda class on vertex 0
    # Tails (indices 1 to i): Identity (fundclass)
    decorations = [lambdaclass(d - i, g - i, n + i)] + [fundclass(1, 1)] * i
    
    norm = 1 / factorial(i)
    return norm * G.boundary_pushforward(decorations)

In [49]:
from itertools import product

def boundary_pushforward_optimized(G, edge_indices):
    """
    Robust optimized version.
    Manually expands terms, filters out zero classes, converts to decstratum,
    and pushes forward in one go.
    """
    
    # --- HELPER: Unwrap TautologicalClass to decstratum ---
    def to_dec(taut_class):
        # We assume the class is non-zero (filtered before calling this)
        return list(taut_class._terms.values())[0]

    # --- 1. Pre-computation ---
    edge_data = []
    all_edges = G.edges()

    for edge_idx in edge_indices:
        edge = all_edges[edge_idx] 
        l1, l2 = edge[0], edge[1]
        
        # Find Vertex 1
        v1 = next(v for v in range(G.num_verts()) if l1 in G.legs(v))
        idx1 = G.legs(v1).index(l1) + 1
        psi1 = -psiclass(idx1, G.genera(v1), G.num_legs(v1))
        
        # Find Vertex 2
        v2 = next(v for v in range(G.num_verts()) if l2 in G.legs(v))
        idx2 = G.legs(v2).index(l2) + 1
        psi2 = -psiclass(idx2, G.genera(v2), G.num_legs(v2))
        
        edge_data.append({
            'v1': v1, 'psi1': psi1,
            'v2': v2, 'psi2': psi2
        })

    # --- 2. Build terms ---
    all_terms = []

    for bits in product([0, 1], repeat=len(edge_indices)):
        
        # Start with Identity (fundclass) on all vertices
        current_term_classes = [fundclass(G.genera(v), G.num_legs(v)) for v in range(G.num_verts())]
        
        # Multiply in the psi classes
        for i, bit in enumerate(bits):
            data = edge_data[i]
            if bit == 0:
                v = data['v1']
                current_term_classes[v] = current_term_classes[v] * data['psi1']
            else:
                v = data['v2']
                current_term_classes[v] = current_term_classes[v] * data['psi2']
        
        # --- CRITICAL FIX: Filter out zero terms ---
        # If any vertex class vanished (e.g. psi^2 on dim 1), the whole term is 0.
        # In admcycles, a zero class has an empty _terms dictionary.
        if any(len(c._terms) == 0 for c in current_term_classes):
            continue 

        # CONVERT to decstratums
        current_term_decs = [to_dec(c) for c in current_term_classes]
        all_terms.append(current_term_decs)

    # --- 3. Create & Pushforward ---
    # If all terms vanished, return 0
    if not all_terms:
        return TautologicalClass(G.to_mean_graph().parent(), {})

    big_class = prodtautclass(G, all_terms)
    return big_class.pushforward()

In [ ]:



# --- 3. The Verification ---

# Now you can just use indices! 
# Example: G[3] is the graph with 3 edges.

# Calculate the bicolored terms
# G[3] has 3 edges (0, 1, 2). We want the first two: [0, 1]
G2Bic = boundary_pushforward_optimized(G[2], [0,1])
G3Bic = boundary_pushforward_optimized(G[3], [0])

# G[4] has 4 edges. We want the first one: [0]
#G4Bic = boundary_pushforward_optimized(G[4], [0])

# Verify the relation: GLU2 * GLU3 - ...
#result = (GLU[2] * GLU[3]) - (6 * G3Bic + 6 * G4Bic + GLU[5])

result = (GLU[2] * GLU[2]) - (2*boundary_pushforward_optimized(G[2],[0,1])+4*boundary_pushforward_optimized(G[3],[0])+GLU[4])

print("Is the relation zero?", result.is_zero())

Is the relation zero? True


In [68]:
from itertools import product # --- 3. The Generalized Multiplier ---

def multiply_glu_terms(g, i, j, k, n=1):
    """
    Computes the contribution to the product of:
      Blue Term: pslambdaclass_term(d=g, i)
      Red Term:  OtherSide_term(j)
    Intersecting along 'k' edges.
    """
    
    # --- GUARD CLAUSE: Impossible Intersections ---
    # We cannot overlap more edges than exist in the source graphs.
    if k > i or k > j or k < 0:
        return 0
    
    # 1. Graph Geometry
    # The resulting graph has (i + j - k) edges
    total_edges = i + j - k
    G = make_glu_graph(g, total_edges, n)
    
    # 2. Edges Logic
    # We map the indices 0..total_edges-1 to their roles:
    # 0 to k-1:          BICOLORED (Overlap)
    # k to j-1:          RED ONLY  (From OtherSide)
    # j to total_edges-1: BLUE ONLY (From pslambda)
    
    # 3. Lambda Distribution
    # Main vertex degree check
    deg_main = (g - i) - (j - k)
    if deg_main < 0:
        return 0 
        
    lambda_main = lambdaclass(deg_main, g - total_edges, n + total_edges)
    lambda_tail = lambdaclass(1, 1, 1) 
    
    # 4. Build Edge Data
    edge_data = []
    
    # We need data for ALL edges in the resulting graph (0 to total_edges-1)
    for idx in range(total_edges):
        l_star = n + 1 + idx
        idx_star = G.legs(0).index(l_star) + 1
        psi_star = psiclass(idx_star, g - total_edges, n + total_edges)
        
        # Tail leg is always index 1
        psi_dot = psiclass(1, 1, 1)
        
        edge_data.append({'psi_star': psi_star, 'psi_dot': psi_dot})

    # 5. Expand Polynomials
    # We loop over the 'j' edges coming from the Red Graph (OtherSide)
    # These cover indices 0 to j-1.
    all_terms = []
    
    for bits in product([0, 1], repeat=j):
        current_classes = [fundclass(1,1) for _ in range(G.num_verts())]
        current_classes[0] = lambda_main
        
        # Apply Lambda_1 to Red-Only Tails (indices k to j-1)
        for red_idx in range(k, j):
             current_classes[red_idx + 1] = lambda_tail
             
        sign = 1
        valid_term = True

        for bit_idx, bit in enumerate(bits):
            edge_idx = bit_idx 
            data = edge_data[edge_idx]
            
            if edge_idx < k:
                # --- BICOLORED (0 to k-1) ---
                if bit == 0: 
                    current_classes[edge_idx + 1] *= (data['psi_dot'] * data['psi_dot'])
                else: 
                    current_classes[0] *= (data['psi_star'] * data['psi_star'])
                    sign *= -1
            else:
                # --- RED ONLY (k to j-1) ---
                if bit == 0:
                    current_classes[edge_idx + 1] *= data['psi_dot']
                else:
                    current_classes[0] *= data['psi_star']
                    sign *= -1

        if any(len(c._terms) == 0 for c in current_classes):
            continue

        decs = [list(c._terms.values())[0] for c in current_classes]
        
        if sign == -1:
             decs[0].poly.coeff = [-c for c in decs[0].poly.coeff]
             
        all_terms.append(decs)

    if not all_terms:
        return 0
    
    return prodtautclass(G, all_terms).pushforward()

In [ ]:
g = 4
n = 1
i = 2
j = 2

# Terms for k = 2, 1, 0
# Coefficients are (i choose k) * (j choose k) * k!
# For i=2, j=2:
# k=2: (1)*(1)*2 = 2
# k=1: (2)*(2)*1 = 4
# k=0: (1)*(1)*1 = 1

term_k2 = 2 * multiply_glu_terms(g, i, j, 2, n) # Overlap 2 -> Result G[2]
term_k1 = 4 * multiply_glu_terms(g, i, j, 1, n) # Overlap 1 -> Result G[3]
term_k0 = 1 * multiply_glu_terms(g, i, j, 0, n) # Overlap 0 -> Result G[4]

RHS = term_k2 + term_k1 + term_k0
LHS = pslambdaclass_term(g, g, n, i) * OtherSide_term(g, n, j)

result = LHS - RHS
print("Is the relation zero?", result.is_zero())